# Phase 4: Evaluation & TensorBoard Analysis

## Why Not Just Use Accuracy?
With 95% real / 5% fake data:
- A model predicting 'real' for everything = **95% accuracy, 0% fraud detection**.
- That's worse than useless for our actual goal.

## Metrics We Care About
| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **Precision** | TP / (TP + FP) | Of jobs flagged as fake, how many actually were? |
| **Recall** | TP / (TP + FN) | Of all fake jobs, how many did we catch? |
| **F1** | 2 × P × R / (P + R) | Harmonic mean of both — the balanced score |
| **AUC-ROC** | Area under ROC curve | Overall discriminative ability |

**For fraud detection, Recall on the fake class matters most** — missing a fake job is worse than a false alarm.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
from torch.utils.tensorboard import SummaryWriter
import io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load best model ────────────────────────────────────────────────────────
model     = DistilBertForSequenceClassification.from_pretrained('../models/best_model')
tokenizer = DistilBertTokenizer.from_pretrained('../models/best_model')
model     = model.to(device)
model.eval()
print('Model loaded.')

In [ ]:
# ── Rebuild test DataLoader ────────────────────────────────────────────────
MAX_LEN    = 256
BATCH_SIZE = 16
SEED       = 42

df = pd.read_csv('../data/processed.csv')
df['combined_text'] = df['combined_text'].fillna('').astype(str)

from sklearn.model_selection import train_test_split
_, temp_df = train_test_split(df, test_size=0.2, stratify=df['fraudulent'], random_state=SEED)
_, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['fraudulent'], random_state=SEED)

class JobPostingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

test_dataset = JobPostingDataset(test_df['combined_text'], test_df['fraudulent'], tokenizer, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# ── Run inference on test set ──────────────────────────────────────────────
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)[:, 1]  # Probability of being fake
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))
print(f'AUC-ROC: {roc_auc_score(all_labels, all_probs):.4f}')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Real', 'Predicted Fake'],
            yticklabels=['Actual Real', 'Actual Fake'])
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (Real correctly identified): {tn}")
print(f"False Positives (Real flagged as Fake)     : {fp}")
print(f"False Negatives (Fake missed!)             : {fn}")
print(f"True Positives  (Fake correctly caught)    : {tp}")

In [ ]:
# ── Precision-Recall Curve ─────────────────────────────────────────────────
# More informative than ROC for imbalanced datasets
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)
avg_precision = average_precision_score(all_labels, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PR Curve
axes[0].plot(recall, precision, color='purple', lw=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve (AP={avg_precision:.3f})')
axes[0].grid(True)

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score   = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1], 'k--', label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../data/pr_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Misclassification Analysis ─────────────────────────────────────────────
# Understanding what we get wrong is as important as knowing what we get right.
test_df = test_df.reset_index(drop=True)
test_df['predicted'] = all_preds
test_df['fake_prob'] = all_probs

# False Negatives: Fake jobs we MISSED (most dangerous error)
false_negatives = test_df[(test_df['fraudulent'] == 1) & (test_df['predicted'] == 0)]
print(f"False Negatives (missed fakes): {len(false_negatives)}")
print('\nSample missed fake job (first 300 chars of text):')
if len(false_negatives) > 0:
    print(false_negatives.iloc[0]['combined_text'][:300])

print()
# False Positives: Real jobs flagged as fake
false_positives = test_df[(test_df['fraudulent'] == 0) & (test_df['predicted'] == 1)]
print(f"False Positives (real flagged as fake): {len(false_positives)}")

In [ ]:
# ── Log final metrics to TensorBoard ──────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

writer = SummaryWriter('../tensorboard_logs/test_evaluation')

writer.add_scalar('Test/Accuracy',  accuracy_score(all_labels, all_preds))
writer.add_scalar('Test/F1_macro',  f1_score(all_labels, all_preds, average='macro'))
writer.add_scalar('Test/Precision', precision_score(all_labels, all_preds, average='macro', zero_division=0))
writer.add_scalar('Test/Recall',    recall_score(all_labels, all_preds, average='macro', zero_division=0))
writer.add_scalar('Test/AUC_ROC',   roc_auc_score(all_labels, all_probs))

# Log confusion matrix image
cm_fig = plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.tight_layout()
writer.add_figure('Confusion_Matrix/test', cm_fig)

writer.close()
print('All test metrics logged to TensorBoard.')
print('Run: tensorboard --logdir ../tensorboard_logs')